In [5]:
import os, subprocess, sys
#local imports
pymipl_path = os.path.abspath('../')
sys.path.append(pymipl_path)
sys.path.append( os.path.abspath(pymipl_path+'/xnat_workflow') ) 
from dicom_sort import *
from pathlib import Path


In [6]:
import csv
#read the scan type mappings.
with open('/workspace/mmilchenko/BM_WU/washu_index_four_mod.csv') as f:
    sessions=list(csv.DictReader(f))

In [7]:
sessions[0]

{'ind': '0',
 'Subject': 'M79614744',
 'Session Label': 'M79614744_20201110144849',
 'id_t2f': '4',
 'series_description_t2f': 'TRA FLAIR NEW',
 'frames_t2f': '27',
 'id_t1ce': '17',
 'series_description_t1ce': 'TRA 3D T1 GRE++STRAIGHT++_ND',
 'frames_t1ce': '192',
 'id': '2',
 'series_description': 'SAG T1',
 'frames': '25',
 'id_t2w': '3',
 'series_description_t2w': 'TRA T2',
 'frames_t2w': '27'}

In [30]:
#define global variables and helper functions. 
import importlib
import workflow_adapters as wa
importlib.reload(wa)

import logging
import datetime
import yaml

def set_logger():
    root = logging.getLogger()
    root.setLevel(logging.DEBUG)
    
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    root.addHandler(handler)

#helper functions
def paths_to_str(x):
    if isinstance(x, Path): return str(x)
    if isinstance(x, dict): return {k: paths_to_str(v) for k, v in x.items()}
    if isinstance(x, list): return [paths_to_str(v) for v in x]
    return x

def resource_to_xnat(local_resource, xnat_session_resource, xnat_project, xnat_subject, xnat_experiment):
    return wa.sync_resource_xnat(local_resource, xnat_session_resource, xnat_project, xnat_subject, xnat_experiment, \
        level="experiment", create_hierarchy=True)

global_vars={}
env_type='jupyter'
env_type='container'
project='BM_WU'
workflow_id='BRATS_preproc'
global_vars['g_workflow_id']=workflow_id
root_dir=Path('/workspace/mmilchenko')

if env_type=='jupyter':
    pymipl_dir=root_dir / "pymipl"

    #path that mounts directory with XNAT experiments
    input_mount_path=Path('/data/projects/BM_WU/experiments')
    local_workdir_path=Path('/workspace/mmilchenko/BM_WU')
    
    global_vars['g_input_mount_path']=input_mount_path
    #path to local directory where processing will be stored
    global_vars['g_local_workdir_path']=local_workdir_path
    #library locations, algorithm specific
    global_vars['g_pymipl_dir']=pymipl_dir
    #main algorithm repository dir
    global_vars['g_env_repo_dir']=root_dir / 'envs/brats'
    #global_vars['g_alg_repo_dir']
    global_vars['g_project']=project
        
elif env_type == 'container': #built-in defaults used in the bootstrap image.
    wa.init_global_vars_bootstrap_image(global_vars,project)


In [31]:
#loop over sessions to create batch scripts.
dt=datetime.datetime.now().strftime("%Y%m%d_%H%M")
batch_file=local_workdir_path / f"batch_{dt}.sh"

for session in sessions:
    job,steps={},[]
    job_experiment=session['Session Label']
    job_scan_context=global_vars['g_input_mount_path'] #/ job_scan_id / 'DICOM'
    if env_type == 'jupyter': job_scan_context = job_scan_context / Path(job_experiment)
    job_scan_context = job_scan_context / Path('SCANS')    
    
    job['job_title']=f"Workflow {workflow_id}, subject {session['Subject']}, experiment {session['Session Label']}"
    job['job_t1w_path'] = job_scan_context / session['id'] / 'DICOM'
    job['job_t1wce_path'] = job_scan_context / session['id_t1ce'] / 'DICOM'
    job['job_t2w_path'] = job_scan_context / session['id_t2w'] / 'DICOM'
    job['job_t2f_path'] = job_scan_context / session['id_t2f'] / 'DICOM'
    job['job_subject'] = session['Subject']
    job['job_exp_label'] =session['Session Label']
    job_id=f"{workflow_id}_{session['Subject']}_{session['Session Label']}"
    job['job_id']=job_id
    
    #directory where the job writes local files. This line must work for both types of workflow.
    job['job_workdir']=global_vars['g_local_workdir_path'] / session['Subject'] / session['Session Label']
         
    #Step 1. Run BRATS preprocessing. 
    step={"step_title": "Run BRATS preprocessing"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/brats_preprocess.py --patient_id {job_subject} \
        --outdir {job_workdir} --t1 {job_t1w_path} --t1ce {job_t1wce_path} --t2 {job_t2w_path} --flair {job_t2f_path}"
    steps+=[step]

    #Steps 2-5. Generate QC images.    
    step={"step_title": "T1w QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/t1_qc.png {job_workdir}/raw/t1.nii.gz"
    steps+=[step]

    step={"step_title": "T1ce QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/t1ce_qc.png {job_workdir}/raw/t1ce.nii.gz"
    steps+=[step]
    
    step={"step_title": "T2w QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/t2_qc.png {job_workdir}/raw/t2.nii.gz"
    steps+=[step]

    step={"step_title": "FLAIR QC image"}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/slice_qc.py \
        --mask {job_workdir}/bet/brain_mask.nii.gz -o {job_workdir}/qc/flair_qc.png {job_workdir}/raw/flair.nii.gz"
    steps+=[step]

    #Step 6. Cleanup.
    step={ "step_title": "Cleanup symlinks" }
    step['step_command']="rm -f {job_workdir}/bet/brain_mask.nii.gz"
    steps+=[step]

    #Step 7. Upload results to session resource. Test for notebook mode only. Container service mode does this internally.
    if env_type=='jupyter':
        step={ "step_title": "Upload results" }
        step['step_command']='micromamba run -p {g_env_repo_dir} python {g_pymipl_dir}/xnat_workflow/sync-resource-with-xnat.py \
            --level experiment --project {g_project} --subject {job_subject} --experiment {job_exp_label} --local_resource {job_workdir} \
            --remote_resource {job_id} --create_hierarchy 1'        
        steps+=[step]
    
    #process job and write out
    job['steps']=steps
    job_id=f"{workflow_id}_{session['Subject']}_{job_experiment}"
    job_dir=local_workdir_path / 'jobs' / job_id
    job_file_yaml=job_dir / 'job.yaml'
    job_file_sh=job_dir / 'job.sh'
    job_dir.mkdir(parents=True,exist_ok=True)
    
    with open(job_file_yaml,"w") as f:
        yaml.safe_dump(paths_to_str(job),f,sort_keys=False)

    if env_type=='jupyter': #write all commands to a single batch file
        workflow_to_batch(job,global_vars,batch_file)
        print(batch_file)
        
    else: #one batch file per job
        #create batch        
        print(job_file_sh)
        #reset job script
        ! truncate -s 0 {job_file_sh}
        #generate job script
        workflow_to_batch(job,global_vars,job_file_sh)
        ! chmod +x {job_file_sh}
        #store batch file.
        print('sending batch to xnat resource')
        res=0
        #res=resource_to_xnat(job_dir, job_id, project, job_subject, job_exp_label)
        if res == 0:
            print('Success')
        else: 
            print (f'Failed sending configuration to session resource, error {res}')
            
    break #debug: do just one run.

/workspace/mmilchenko/BM_WU/jobs/BRATS_preproc_M79614744_M79614744_20201110144849/job.sh
sending batch to xnat resource
Success
